## DSAN6000 Week 10: Distributed ML Pipelines with Spark MLlib and MLFlow

First things first, we establish our `SparkSession` object that we'll need if we want to distribute our tasks across a cluster!

In [ ]:
!sudo apt update

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,733 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,170 

In [ ]:
!sudo apt install openjdk-21-jdk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-common at-spi2-core fonts-dejavu-core fonts-dejavu-extra
  fonts-dejavu-mono gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0t64 libatspi2.0-0t64
  libgtk-3-0t64 libgtk-3-bin libgtk-3-common librsvg2-common libxcomposite1
  libxt-dev libxtst6 libxxf86dga1 openjdk-21-jre session-migration x11-utils
Suggested packages:
  gvfs libxt-doc openjdk-21-demo openjdk-21-source visualvm mesa-utils
Recommended packages:
  luit
The following NEW packages will be installed:
  at-spi2-common at-spi2-core fonts-dejavu-core fonts-dejavu-extra
  fonts-dejavu-mono gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0t64 libatspi2.0-0t64
  libgtk-3-0t64 libgtk-3-bin libgtk-3-common librsvg2-common libxcomposite1
  libxt-dev libxtst6 libxxf86

In [ ]:
!java --version

[0.020s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.020s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk 21.0.12.1 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('SF_Housing')
    .getOrCreate()
)

In [ ]:
spark

## Random Forest Classification with Spark MLlib

In [ ]:
import pandas as pd

In [ ]:
def my_rep(thing):
    return '<%s.%s object at %s>' % (
        thing.__class__.__module__,
        thing.__class__.__name__,
        hex(id(thing))
    )

In [ ]:
my_df = pd.DataFrame({'x': [1,2,3], 'y': [4, 5, 6]})
my_df

,x,y
0,1,4
1,2,5
2,3,6


In [ ]:
my_rep(my_df)

'<pandas.core.frame.DataFrame object at 0x7a64f7f9e660>'

In [ ]:
from pyspark.ml import Pipeline

In [ ]:
from pyspark.ml.feature import VectorAssembler

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
!wget https://jpj.georgetown.domains/dsan6000-data/sf-airbnb-clean.parquet.zip
!unzip sf-airbnb-clean.parquet.zip

--2025-10-30 23:35:13--  https://jpj.georgetown.domains/dsan6000-data/sf-airbnb-clean.parquet.zip
Resolving jpj.georgetown.domains (jpj.georgetown.domains)... 138.197.74.192
Connecting to jpj.georgetown.domains (jpj.georgetown.domains)|138.197.74.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 141892 (139K) [application/zip]
Saving to: ‘sf-airbnb-clean.parquet.zip’

sf-airbnb-clean.par 100%[===================>] 138.57K   238KB/s    in 0.6s    

2025-10-30 23:35:15 (238 KB/s) - ‘sf-airbnb-clean.parquet.zip’ saved [141892/141892]

Archive:  sf-airbnb-clean.parquet.zip
   creating: sf-airbnb-clean.parquet/
  inflating: __MACOSX/._sf-airbnb-clean.parquet  
  inflating: sf-airbnb-clean.parquet/_committed_4320459746949313749  
  inflating: __MACOSX/sf-airbnb-clean.parquet/.__committed_4320459746949313749  
 extracting: sf-airbnb-clean.parquet/_started_4320459746949313749  
  inflating: __MACOSX/sf-airbnb-clean.parquet/.__started_4320459746949313749  
  inflatin

In [ ]:
# Lazy evaluation
abb_fpath = "sf-airbnb-clean.parquet"
abb_df = (
    spark
    .read
    .parquet(abb_fpath)
)
abb_df

DataFrame[host_is_superhost: string, cancellation_policy: string, instant_bookable: string, host_total_listings_count: double, neighbourhood_cleansed: string, latitude: double, longitude: double, property_type: string, room_type: string, accommodates: double, bathrooms: double, bedrooms: double, beds: double, bed_type: string, minimum_nights: double, number_of_reviews: double, review_scores_rating: double, review_scores_accuracy: double, review_scores_cleanliness: double, review_scores_checkin: double, review_scores_communication: double, review_scores_location: double, review_scores_value: double, price: double, bedrooms_na: double, bathrooms_na: double, beds_na: double, review_scores_rating_na: double, review_scores_accuracy_na: double, review_scores_cleanliness_na: double, review_scores_checkin_na: double, review_scores_communication_na: double, review_scores_location_na: double, review_scores_value_na: double]

In [ ]:
# Eager evaluation
abb_df.show(n=4)

+-----------------+--------------------+----------------+-------------------------+----------------------+--------+----------+-------------+---------------+------------+---------+--------+----+--------+--------------+-----------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+-----+-----------+------------+-------+-----------------------+-------------------------+----------------------------+------------------------+------------------------------+-------------------------+----------------------+
|host_is_superhost| cancellation_policy|instant_bookable|host_total_listings_count|neighbourhood_cleansed|latitude| longitude|property_type|      room_type|accommodates|bathrooms|bedrooms|beds|bed_type|minimum_nights|number_of_reviews|review_scores_rating|review_scores_accuracy|review_scores_cleanliness|review_scores_checkin|review_scores_communication|review_scores_location|

In [ ]:
abb_df.select('neighbourhood_cleansed').distinct().show(n=50)

+----------------------+
|neighbourhood_cleansed|
+----------------------+
|          Inner Sunset|
|        Haight Ashbury|
|    Financial District|
|           North Beach|
|      Western Addition|
|  Downtown/Civic Ce...|
|        Bernal Heights|
|             Lakeshore|
|          Outer Sunset|
|     Visitacion Valley|
|        Inner Richmond|
|              Nob Hill|
|        Outer Richmond|
|        Crocker Amazon|
|             Chinatown|
|               Bayview|
|               Mission|
|             Excelsior|
|            Twin Peaks|
|              Seacliff|
|      Presidio Heights|
|      Golden Gate Park|
|             Glen Park|
|              Presidio|
|          Potrero Hill|
|              Parkside|
|         Outer Mission|
|   Castro/Upper Market|
|                Marina|
|            Noe Valley|
|       Diamond Heights|
|            Ocean View|
|          Russian Hill|
|       South of Market|
|       Pacific Heights|
|    West of Twin Peaks|
+----------------------+


In [ ]:
abb_df.show(n=2, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------
 host_is_superhost              | t                           
 cancellation_policy            | moderate                    
 instant_bookable               | t                           
 host_total_listings_count      | 1.0                         
 neighbourhood_cleansed         | Western Addition            
 latitude                       | 37.76931                    
 longitude                      | -122.43386                  
 property_type                  | Apartment                   
 room_type                      | Entire home/apt             
 accommodates                   | 3.0                         
 bathrooms                      | 1.0                         
 bedrooms                       | 1.0                         
 beds                           | 2.0                         
 bed_type                       | Real Bed                    
 minimum_nights                 | 1.0                  

In [ ]:
(train_df, test_df) = abb_df.randomSplit([.8, .2], seed=6000)
print(f"""There are {train_df.count()} rows in the training set, and {test_df.count()} in the test set""")

There are 5731 rows in the training set, and 1415 in the test set


In [ ]:
# Handle string category columns
categoricalCols = [
    field
    for (field, dataType) in train_df.dtypes
    if dataType == "string"
]

indexOutputCols = [x + "Index" for x in categoricalCols]

stringIndexer = StringIndexer(
    inputCols=categoricalCols,
    outputCols=indexOutputCols,
    handleInvalid="skip"
)

# Handle numeric columns
numericCols = [
    field
    for (field, dataType) in train_df.dtypes
    if ((dataType == "double") & (field != "price"))
]

In [ ]:
# Lazy evaluation
assemblerInputs = indexOutputCols + numericCols
vecAssembler = VectorAssembler(
    inputCols=assemblerInputs,
    outputCol="features"
)
assembler_pipeline = Pipeline(
    stages=[
        stringIndexer,
        vecAssembler
    ]
)
print(assembler_pipeline.explainParams())

stages: a list of pipeline stages (current: [StringIndexer_56dd3f29a13d, VectorAssembler_31ea05fc2a7c])


In [ ]:
# Lazy evaluation [*optimized internally]
fitted_assembler_pl = assembler_pipeline.fit(train_df)
fitted_assembler_pl

PipelineModel_ea2586483bf8

In [ ]:
# Lazy evaluation
fitted_assembler_result = fitted_assembler_pl.transform(train_df)
fitted_assembler_result

DataFrame[host_is_superhost: string, cancellation_policy: string, instant_bookable: string, host_total_listings_count: double, neighbourhood_cleansed: string, latitude: double, longitude: double, property_type: string, room_type: string, accommodates: double, bathrooms: double, bedrooms: double, beds: double, bed_type: string, minimum_nights: double, number_of_reviews: double, review_scores_rating: double, review_scores_accuracy: double, review_scores_cleanliness: double, review_scores_checkin: double, review_scores_communication: double, review_scores_location: double, review_scores_value: double, price: double, bedrooms_na: double, bathrooms_na: double, beds_na: double, review_scores_rating_na: double, review_scores_accuracy_na: double, review_scores_cleanliness_na: double, review_scores_checkin_na: double, review_scores_communication_na: double, review_scores_location_na: double, review_scores_value_na: double, host_is_superhostIndex: double, cancellation_policyIndex: double, instan

In [ ]:
# Eager evaluation
fitted_assembler_result.show(n=3, truncate=False)

+-----------------+-------------------+----------------+-------------------------+----------------------+--------+----------+-------------+---------------+------------+---------+--------+----+--------+--------------+-----------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+-----+-----------+------------+-------+-----------------------+-------------------------+----------------------------+------------------------+------------------------------+-------------------------+----------------------+----------------------+------------------------+---------------------+---------------------------+------------------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|host_is_superhost|cancellation_policy|instant_bookable|host_total_listings_c

In [ ]:
fitted_assembler_result.show(n=2, vertical=True, truncate=False)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 host_is_superhost              | f                                                                                                                                                           
 cancellation_policy            | flexible                                                                                                                                                    
 instant_bookable               | f                                                                                                                                                           
 host_total_listings_count      | 0.0                                                                                                                                                         
 neighbourhood_cleansed         | Diamond Hei

In [ ]:
rf = RandomForestRegressor(
    labelCol="price",
    maxBins=40,
    maxDepth=5,
    numTrees=10,
    seed=6000
)
print(rf.explainParams())

bootstrap: Whether bootstrap samples are used when building trees. (default: True)
cacheNodeIds: If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval. (default: False)
checkpointInterval: set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext. (default: 10)
featureSubsetStrategy: The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'onethird' for regression), 'all' (use all features), 'onethird' (use 1/3 of the featur

In [ ]:
pipeline = Pipeline(
    stages=[
        stringIndexer,
        vecAssembler,
        rf
    ]
)
print(pipeline.explainParams())

stages: a list of pipeline stages (current: [StringIndexer_56dd3f29a13d, VectorAssembler_31ea05fc2a7c, RandomForestRegressor_71d322c73441])


In [ ]:
# Lazy evaluation
fitted_pipeline = pipeline.fit(train_df)
fitted_pipeline

PipelineModel_7321dd3ecd50

In [ ]:
# Lazy evaluation
test_result = fitted_pipeline.transform(test_df)
test_result

DataFrame[host_is_superhost: string, cancellation_policy: string, instant_bookable: string, host_total_listings_count: double, neighbourhood_cleansed: string, latitude: double, longitude: double, property_type: string, room_type: string, accommodates: double, bathrooms: double, bedrooms: double, beds: double, bed_type: string, minimum_nights: double, number_of_reviews: double, review_scores_rating: double, review_scores_accuracy: double, review_scores_cleanliness: double, review_scores_checkin: double, review_scores_communication: double, review_scores_location: double, review_scores_value: double, price: double, bedrooms_na: double, bathrooms_na: double, beds_na: double, review_scores_rating_na: double, review_scores_accuracy_na: double, review_scores_cleanliness_na: double, review_scores_checkin_na: double, review_scores_communication_na: double, review_scores_location_na: double, review_scores_value_na: double, host_is_superhostIndex: double, cancellation_policyIndex: double, instan

In [ ]:
# Eager evaluation
test_result.show(n=10)

+-----------------+-------------------+----------------+-------------------------+----------------------+--------+----------+-------------+---------------+------------+---------+--------+----+-------------+--------------+-----------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+-----+-----------+------------+-------+-----------------------+-------------------------+----------------------------+------------------------+------------------------------+-------------------------+----------------------+----------------------+------------------------+---------------------+---------------------------+------------------+--------------+-------------+--------------------+------------------+
|host_is_superhost|cancellation_policy|instant_bookable|host_total_listings_count|neighbourhood_cleansed|latitude| longitude|property_type|      room_type|accommodates|bathrooms|bedrooms|b

In [ ]:
rmse_evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)
print(rmse_evaluator.explainParams())

labelCol: label column name. (default: label, current: price)
metricName: metric name in evaluation - one of:
                       rmse - root mean squared error (default)
                       mse - mean squared error
                       r2 - r^2 metric
                       mae - mean absolute error
                       var - explained variance. (default: rmse, current: rmse)
predictionCol: prediction column name. (default: prediction, current: prediction)
throughOrigin: whether the regression is through the origin. (default: False)
weightCol: weight column name. If this is not set or empty, we treat all instance weights as 1.0. (undefined)


In [ ]:
# Eager evaluation
rmse_evaluator.evaluate(test_result)

174.3086109710233

In [ ]:
finisher_pipeline = Pipeline(
    stages=[
        stringIndexer,
        vecAssembler,
        rf,
    ]
)
print(finisher_pipeline.explainParams())

stages: a list of pipeline stages (current: [StringIndexer_56dd3f29a13d, VectorAssembler_31ea05fc2a7c, RandomForestRegressor_71d322c73441])


In [ ]:
finisher_pipeline.fit(train_df)

TypeError: Cannot recognize a pipeline stage of type <class 'pyspark.ml.evaluation.RegressionEvaluator'>.

### Evaluation in a Grid

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [2, 4])
    .addGrid(rf.numTrees, [10, 20])
    .build()
)

In [ ]:
type(param_grid[0])
print(param_grid[0])

{Param(parent='RandomForestRegressor_71d322c73441', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2}


In [ ]:
from pyspark.ml.tuning import CrossValidator
cv = CrossValidator(
    estimator=pipeline,
    evaluator=rmse_evaluator,
    estimatorParamMaps=param_grid,
    numFolds=3,
    seed=6000
)
cv_model = cv.fit(train_df)
cv_model

CrossValidatorModel_02d6eb6c1783

In [ ]:
cv_model.getEstimatorParamMaps()

[{Param(parent='RandomForestRegressor_71d322c73441', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2},
 {Param(parent='RandomForestRegressor_71d322c73441', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 4}]

In [ ]:
list(zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics))

[({Param(parent='RandomForestRegressor_71d322c73441', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 2},
  np.float64(311.40613184487887)),
 ({Param(parent='RandomForestRegressor_71d322c73441', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 4},
  np.float64(309.6154625794024))]

In [ ]:
cv_model = cv.setParallelism(4).fit(trainDF)

In [ ]:
cv = CrossValidator(
    estimator=rf,
    evaluator=evaluator,
    estimatorParamMaps=param_grid,
    numFolds=3,
    parallelism=4,
    seed=42
)

pipeline = Pipeline(
    stages=[
        stringIndexer,
        vecAssembler,
        cv
    ]
)
pipeline_model = pipeline.fit(train_df)


In [ ]:
pipeline_model.extractParamMap()

{}

In [ ]:
test_evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)

In [ ]:
test_evaluator.evaluate(test_df)

IllegalArgumentException: prediction does not exist. Available: host_is_superhost, cancellation_policy, instant_bookable, host_total_listings_count, neighbourhood_cleansed, latitude, longitude, property_type, room_type, accommodates, bathrooms, bedrooms, beds, bed_type, minimum_nights, number_of_reviews, review_scores_rating, review_scores_accuracy, review_scores_cleanliness, review_scores_checkin, review_scores_communication, review_scores_location, review_scores_value, price, bedrooms_na, bathrooms_na, beds_na, review_scores_rating_na, review_scores_accuracy_na, review_scores_cleanliness_na, review_scores_checkin_na, review_scores_communication_na, review_scores_location_na, review_scores_value_na

## Reproducible ML with MLFlow

In [ ]:
%pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.6/752.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 13.4 MB/s eta 0:00:00


In [ ]:
import mlflow
import mlflow.spark
import pandas as pd

In [ ]:
with mlflow.start_run(run_name="random-forest") as run:
  # Log params: num_trees and max_depth
  mlflow.log_param("num_trees", rf.getNumTrees())
  mlflow.log_param("max_depth", rf.getMaxDepth())

  # Log model
  pipelineModel = pipeline.fit(train_df)
  mlflow.spark.log_model(pipelineModel, "model")

  # Log metrics: RMSE and R2
  predDF = pipelineModel.transform(test_df)
  regressionEvaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price")
  rmse = regressionEvaluator.setMetricName("rmse").evaluate(predDF)

  r2 = regressionEvaluator.setMetricName("r2").evaluate(predDF)

  mlflow.log_metrics({"rmse": rmse, "r2": r2})

  # Log artifact: feature importance scores
  rfModel = pipelineModel.stages[-1]
  pandasDF = (
    pd.DataFrame(list(zip(vecAssembler.getInputCols(), rfModel.featureImportances)), columns=["feature", "importance"]) .sort_values(by="importance", ascending=False)
  )

  # First write to local filesystem, then tell MLflow where to find that file
  pandasDF.to_csv("feature-importance.csv", index=False)
  mlflow.log_artifact("feature-importance.csv")


2025/10/27 19:24:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
